# Fine-tuning NLLB-200 for Hindi to Punjabi classroom sentences (LoRA)

**What we do here, in plain words**
1. Start from **NLLB-200 (distilled, 600M)**, a translation model Meta already trained on 200 languages. We do *not* train it from scratch.
2. Build a training set from two sources: (a) a cleaned sample of the public **CoRil-Parallel** Hindi-Punjabi corpus (CC-BY-4.0), and (b) **our own classroom sentences** whose Punjabi was checked by a Punjabi speaker.
3. Measure the model **before** training (baseline) on two test sets: our classroom test sentences and a general-domain test sample.
4. **Fine-tune** with **LoRA**, which trains small extra matrices (about 1 to 2 percent of the weights).
5. Measure again on the **same test sets**, compare, and save the small adapter for the backend (`TRANSLATION_ENGINE=local_model`).

**Before you run:** Runtime menu, Change runtime type, choose **T4 GPU**. Have `pairs.csv` ready (columns `hindi,punjabi,checked`; only rows with `checked = 1` are used).

In [ ]:
!pip -q install -U transformers peft datasets sacrebleu sentencepiece accelerate huggingface_hub

In [ ]:
import os, random, re
import numpy as np, pandas as pd, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE = "facebook/nllb-200-distilled-600M"   # pretrained model we start from
SRC, TGT = "hin_Deva", "pan_Guru"           # NLLB language codes: Hindi (Devanagari), Punjabi (Gurmukhi)
MAX_LEN = 64
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(no GPU: change the runtime type)")

## 1a. General data: CoRil-Parallel (Hindi to Punjabi)
Source: https://huggingface.co/datasets/HimangY/CoRil-Parallel (licence CC-BY-4.0; paper arXiv 2509.19941). The Hindi-Punjabi files are tab-separated lines (`hindi <TAB> punjabi`) in two domains, government and health.

**Be aware:** the government part is state-assembly debates and the translations are noisy (we saw wrong names and odd word choices). So the cleaning step keeps only **short** sentences, drops political and legislative wording, brackets and mixed-script lines, and removes badly misaligned pairs. It still will not be perfect. Read the printed samples and judge the quality yourself; the quality of this data limits what the model can learn.

In [ ]:
import os
import random
import re

import pandas as pd

REPO = "HimangY/CoRil-Parallel"          # CC-BY-4.0, Hindi-Punjabi files are under HI-PA/
_DEV = re.compile(r"[\u0900-\u097F]")     # Devanagari
_GUR = re.compile(r"[\u0A00-\u0A7F]")     # Gurmukhi
_LAT = re.compile(r"[A-Za-z]")
_BAD = re.compile(r"xxx|[(){}\[\]]|--|\.\.|व्यवधान|मेज", re.I)
# The GOV part is state-assembly debates. Drop political / legislative lines: not suitable for a children's tool.
_POLITICS = ("श्री ", "श्रीमती", "माननीय", "अध्यक्ष", "सदन", "मंत्री", "विधेयक", "सरकार", "विधान", "भाजपा",
             "कांग्रेस", "पार्टी", "बजट", "कर्ज", "चुनाव", "विधायक", "प्रश्न", "उत्तर", "प्रतिवेदन")


def _norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[।॥\.\,\?\!\-\"';:()]", " ", str(s))).strip().lower()


def clean_pair(h, p, min_words=3, max_words=14):
    h, p = h.strip(), p.strip()
    if not h or not p or _BAD.search(h) or _BAD.search(p):
        return False
    if any(w in h for w in _POLITICS):
        return False
    if _LAT.search(h) or _LAT.search(p):          # skip mixed-script lines
        return False
    if not _DEV.search(h) or not _GUR.search(p):  # Hindi must be Devanagari, Punjabi Gurmukhi
        return False
    hw, pw = len(h.split()), len(p.split())
    if not (min_words <= hw <= max_words):        # short sentences only
        return False
    return 0.5 <= pw / hw <= 2.0                  # drop badly misaligned pairs


def build_corpus(max_train=6000, max_test=300, seed=42, cache_dir=None):
    """Download CoRil HI-PA, clean it, return (train_df, general_test_df) with columns hindi, punjabi, key."""
    from huggingface_hub import hf_hub_download, list_repo_files
    files = [f for f in list_repo_files(REPO, repo_type="dataset") if f.startswith("HI-PA/") and f.endswith(".txt")]
    if not files:
        raise RuntimeError("No HI-PA files found. Check https://huggingface.co/datasets/HimangY/CoRil-Parallel")
    rows = {"train": [], "test": []}
    for f in files:
        split = "test" if f.endswith("_test.txt") else "train" if f.endswith(("_train.txt", "_dev.txt")) else None
        if split is None:
            continue
        path = hf_hub_download(REPO, f, repo_type="dataset", cache_dir=cache_dir)
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                parts = line.rstrip("\n").split("\t")      # format: hindi <TAB> punjabi
                if len(parts) == 2 and clean_pair(*parts):
                    rows[split].append((parts[0].strip(), parts[1].strip()))

    def frame(pairs):
        df = pd.DataFrame(pairs, columns=["hindi", "punjabi"])
        df["key"] = df.hindi.map(_norm)
        return df.drop_duplicates("key").sample(frac=1, random_state=seed).reset_index(drop=True)

    test = frame(rows["test"]).head(max_test)
    train = frame(rows["train"])
    train = train[~train.key.isin(set(test.key))].head(max_train).reset_index(drop=True)
    return train, test


SEED = SEED if "SEED" in globals() else 42
extra, gen_test = build_corpus(max_train=6000, max_test=300, seed=SEED)
extra[["hindi", "punjabi"]].to_csv("extra.csv", index=False)
gen_test[["hindi", "punjabi"]].to_csv("general_test.csv", index=False)
print(f"general train pairs: {len(extra)} | general test pairs: {len(gen_test)}")
extra.sample(10, random_state=1)[["hindi", "punjabi"]]

## 1b. Our classroom data and the splits
Upload `pairs.csv` (the classroom sentences with Punjabi checked by a Punjabi speaker; column `checked` = 1 for approved rows). We split it **before** any training:
- **test (10%)**: used once at the end for our classroom accuracy
- **dev (10%)**: used during training to detect overfitting
- **train (rest)**: repeated `UPSAMPLE` times and mixed with the general pairs, so the small classroom set is not drowned out by thousands of general sentences

The general-domain test pairs and all dev/test sentences are removed from the training data, and the notebook stops with an error if any sentence appears in two sets.

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose pairs.csv (columns: hindi,punjabi,checked)

In [ ]:
UPSAMPLE = 5   # classroom sentences are repeated this many times in training

def load(path):
    df = pd.read_csv(path).dropna(subset=["hindi", "punjabi"])
    if "checked" in df.columns:                   # only rows a Punjabi speaker approved
        df = df[df["checked"].astype(str).str.strip().isin(["1", "1.0", "yes", "true", "True"])]
    df["key"] = df.hindi.map(_norm)
    return df.drop_duplicates("key")

pairs = load("pairs.csv").sample(frac=1, random_state=SEED).reset_index(drop=True)
n = len(pairs)
n_test, n_dev = max(10, int(0.1 * n)), max(10, int(0.1 * n))
test, dev, cls_train = pairs[:n_test], pairs[n_test:n_test + n_dev], pairs[n_test + n_dev:]

held_out = set(test.key) | set(dev.key)
general = extra[~extra.key.isin(held_out)]
gen_test = gen_test[~gen_test.key.isin(set(general.key) | set(cls_train.key))]
train = pd.concat([general] + [cls_train] * UPSAMPLE).sample(frac=1, random_state=SEED).reset_index(drop=True)

assert not (set(train.key) & set(test.key)) and not (set(train.key) & set(dev.key)), "Data leakage (train vs dev/test)!"
assert not (set(dev.key) & set(test.key)), "Data leakage (dev vs test)!"
print(f"classroom: train {len(cls_train)} (x{UPSAMPLE}) | dev {len(dev)} | test {len(test)}")
print(f"general:   train {len(general)} | test {len(gen_test)}")
print(f"total training rows: {len(train)}")
if len(pairs) < 100:
    print("Warning: fewer than 100 checked classroom pairs. The classroom test set is tiny, so scores will be noisy.")

## 2. Baseline: the pretrained model before any training
We load the model and translate the test sentences. These scores are what we must beat.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import sacrebleu

tok = AutoTokenizer.from_pretrained(BASE, src_lang=SRC, tgt_lang=TGT)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE).to(device)

chrf_pp = sacrebleu.CHRF(word_order=2)   # chrF++ : character-level, works better than BLEU for Indian languages

def translate_all(m, sentences, bs=16):
    m.eval(); out = []
    for i in range(0, len(sentences), bs):
        enc = tok(sentences[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(device)
        with torch.no_grad():
            gen = m.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(TGT), num_beams=4, max_new_tokens=MAX_LEN)
        out += tok.batch_decode(gen, skip_special_tokens=True)
    return out

def score(hyps, refs):
    return {"chrF++": round(chrf_pp.corpus_score(hyps, [refs]).score, 1),
            "BLEU": round(sacrebleu.corpus_bleu(hyps, [refs]).score, 1)}

base_hyps = translate_all(model, test.hindi.tolist())
base_scores = score(base_hyps, test.punjabi.tolist())
print("BASELINE (no training):", base_scores)

base_gen_hyps = translate_all(model, gen_test.hindi.tolist())
base_gen_scores = score(base_gen_hyps, gen_test.punjabi.tolist())
print("BASELINE general-domain test:", base_gen_scores)

## 3. LoRA: train only small adapter matrices
`r=16` is the rank (size) of the adapters. They are added to the attention `q_proj` and `v_proj` layers. The original 600M weights stay frozen,
so training is fast, fits on a free T4 GPU, and is less likely to wipe out what the model already knows.

In [ ]:
from peft import LoraConfig, get_peft_model

cfg = LoraConfig(task_type="SEQ_2_SEQ_LM", r=16, lora_alpha=32, lora_dropout=0.05,
                 target_modules=["q_proj", "v_proj"])
model = get_peft_model(model, cfg)
model.print_trainable_parameters()   # note this number: it is the "only a small percent trained" claim

In [ ]:
from datasets import Dataset

def make(df):
    ds = Dataset.from_pandas(df[["hindi", "punjabi"]].reset_index(drop=True))
    return ds.map(lambda b: tok(b["hindi"], text_target=b["punjabi"], max_length=MAX_LEN, truncation=True),
                  batched=True, remove_columns=["hindi", "punjabi"])

train_ds, dev_ds = make(train), make(dev)
print(train_ds)

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

args = Seq2SeqTrainingArguments(
    output_dir="out",
    per_device_train_batch_size=16, per_device_eval_batch_size=16,
    learning_rate=2e-4,                 # LoRA usually uses a higher rate than full fine-tuning
    num_train_epochs=5,                 # passes over the training data
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    save_total_limit=1, logging_steps=10, fp16=(device == "cuda"),
    report_to="none", seed=SEED)

trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds,
                         data_collator=DataCollatorForSeq2Seq(tok, padding=True))
trainer.train()

for r in trainer.state.log_history:
    if "eval_loss" in r:
        print(f"epoch {r['epoch']:.0f}: dev loss {r['eval_loss']:.3f}")

## 4. Evaluate on the untouched test set
If the dev loss kept falling and the test scores beat the baseline, training helped. If the dev loss rose while the train loss fell, the model was **overfitting**
(memorising the few training sentences). Report whichever result you actually get.

In [ ]:
ft_hyps = translate_all(model, test.hindi.tolist())
ft_scores = score(ft_hyps, test.punjabi.tolist())
ft_gen_hyps = translate_all(model, gen_test.hindi.tolist())
ft_gen_scores = score(ft_gen_hyps, gen_test.punjabi.tolist())
print("CLASSROOM test   BASELINE  :", base_scores)
print("CLASSROOM test   FINE-TUNED:", ft_scores)
print("GENERAL test     BASELINE  :", base_gen_scores)
print("GENERAL test     FINE-TUNED:", ft_gen_scores)

out = pd.DataFrame({"hindi": test.hindi.values, "reference": test.punjabi.values,
                    "baseline": base_hyps, "fine_tuned": ft_hyps})
out.to_csv("test_outputs.csv", index=False)
out.head(10)

## 5. Save the adapter
Only the small adapter is saved (a few MB). The backend loads the same base model and applies this adapter on top.
Unzip `hi-pa-lora.zip` into the project's `models/hi-pa-lora/` folder.

In [ ]:
model.save_pretrained("hi-pa-lora")
import shutil
shutil.make_archive("hi-pa-lora", "zip", "hi-pa-lora")
files.download("hi-pa-lora.zip")
files.download("test_outputs.csv")